# Contagion Simulations

This notebook implements **Option 1** for systemic importance.

Strategy:
- default one bank at a time;
- propagate losses through the interbank network using the default contagion rule;
- measure cascade size, secondary defaults, equity depletion, and affected assets;
- store the essential outputs in parquet tables;
- rank banks by the damage caused by their default.

The notebook now uses the batch workflow implemented in `src/models/contagion.py`:
- `run_default_contagion_analysis(...)`
- `run_default_contagion_analysis_for_quarter(...)`
- `build_systemic_importance_summary(...)`


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '..')

import pandas as pd

from src.data.data_loader import load_data
from src.models.contagion import (
    run_default_contagion_analysis,
    run_default_contagion_analysis_for_quarter,
)


In [ ]:
# Configuration
MECHANISM = 'Exposure'
ALPHA = 1.0
IMPORTANCE_QUANTILE = 0.80
TRACK_ROUNDS = True

PROJECT_ROOT = Path().resolve().parent
DATA_PATH = PROJECT_ROOT / 'datasets'
OUTPUT_ROOT = PROJECT_ROOT / 'src' / 'data'

TABLE_DIRS = {
    'runs': OUTPUT_ROOT / 'sim_runs',
    'bank_state': OUTPUT_ROOT / 'sim_bank_state',
    'round_summary': OUTPUT_ROOT / 'sim_round_summary',
    'importance': OUTPUT_ROOT / 'systemic_importance',
}

for path in TABLE_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print('Output directories:')
for name, path in TABLE_DIRS.items():
    print(f'  {name:13s} -> {path}')


## Single-quarter example

Start with one quarter to inspect the network and run the simulation end to end.

In [ ]:
YEAR = 2023
QUARTER = 1

edges, nodes = load_data(YEAR, QUARTER, data_path=DATA_PATH)

print(f'Quarter: {YEAR}Q{QUARTER}')
print(f'Banks:   {len(nodes):,}')
print(f'Edges:   {len(edges):,}')

display(nodes.head())
display(edges.head())


In [ ]:
example_outputs = run_default_contagion_analysis(
    edges=edges,
    nodes=nodes,
    year=YEAR,
    quarter=QUARTER,
    mechanism=MECHANISM,
    alpha=ALPHA,
    importance_quantile=IMPORTANCE_QUANTILE,
    track_rounds=TRACK_ROUNDS,
    output_dir=OUTPUT_ROOT,
)

runs_df = example_outputs['runs']
bank_state_df = example_outputs['bank_state']
round_summary_df = example_outputs['round_summary']
importance_df = example_outputs['importance']

print('Generated tables:')
print(f"  runs          : {runs_df.shape}")
print(f"  bank_state    : {bank_state_df.shape}")
print(f"  round_summary : {round_summary_df.shape}")
print(f"  importance    : {importance_df.shape}")
print('\nWritten parquet files:')
for key, value in example_outputs.get('files', {}).items():
    print(f'  {key:13s} -> {value}')


## Systemic importance ranking

The ranking table stores the essential indicators used to determine whether a bank is systemically important.

In [ ]:
ranking_cols = [
    'bank_id',
    'cascade_size',
    'secondary_defaults',
    'num_impacted',
    'failed_equity_loss',
    'system_equity_depletion',
    'affected_assets',
    'affected_assets_share',
    'composite_rank',
    'is_systemically_important',
]

display(importance_df[ranking_cols].head(20))

print('Systemically important banks:', int(importance_df['is_systemically_important'].sum()))
print('Share of banks labeled systemic:', importance_df['is_systemically_important'].mean())


In [ ]:
summary_stats = importance_df[
    [
        'cascade_size',
        'secondary_defaults',
        'num_impacted',
        'failed_equity_loss',
        'system_equity_depletion',
        'affected_assets_share',
        'composite_score',
    ]
].describe()

display(summary_stats)


## Parquet validation

Reload the exported tables for the same quarter and run a few consistency checks.

In [ ]:
runs_path = TABLE_DIRS['runs'] / f'sim_runs_{YEAR}Q{QUARTER}.parquet'
bank_state_path = TABLE_DIRS['bank_state'] / f'sim_bank_state_{YEAR}Q{QUARTER}.parquet'
round_summary_path = TABLE_DIRS['round_summary'] / f'sim_round_summary_{YEAR}Q{QUARTER}.parquet'
importance_path = TABLE_DIRS['importance'] / f'systemic_importance_{YEAR}Q{QUARTER}.parquet'

runs_check = pd.read_parquet(runs_path)
bank_state_check = pd.read_parquet(bank_state_path)
round_summary_check = pd.read_parquet(round_summary_path)
importance_check = pd.read_parquet(importance_path)

print(runs_check.shape)
print(bank_state_check.shape)
print(round_summary_check.shape)
print(importance_check.shape)


In [ ]:
# Check 1: one run per initial bank
assert len(runs_check) == len(nodes), 'Expected one simulation run per bank.'

# Check 2: Table 1 num_failed must match the number of failed banks in Table 2
failed_counts = (
    bank_state_check[bank_state_check['failed']]
    .groupby('run_id')
    .size()
    .rename('failed_in_bank_state')
    .reset_index()
)

cross_check = runs_check[['run_id', 'num_failed']].merge(
    failed_counts,
    on='run_id',
    how='left',
)
cross_check['failed_in_bank_state'] = cross_check['failed_in_bank_state'].fillna(0).astype(int)

mismatches = cross_check[cross_check['num_failed'] != cross_check['failed_in_bank_state']]
print('Cross-table mismatches:', len(mismatches))
assert mismatches.empty, 'Tables are inconsistent.'

# Check 3: the ranking table must also have one row per initial bank
assert len(importance_check) == len(nodes), 'Expected one ranking row per bank.'
assert importance_check['bank_id'].is_unique, 'Each bank should appear once in the ranking table.'

print('All validation checks passed.')


## Run all quarters

Uncomment the next cell when you want to generate outputs for the full 2016Q1 to 2023Q4 period.

In [ ]:
# all_quarter_summaries = []
#
# for year in range(2016, 2024):
#     for quarter in range(1, 5):
#         outputs = run_default_contagion_analysis_for_quarter(
#             year=year,
#             quarter=quarter,
#             data_path=DATA_PATH,
#             output_dir=OUTPUT_ROOT,
#             mechanism=MECHANISM,
#             alpha=ALPHA,
#             importance_quantile=IMPORTANCE_QUANTILE,
#             track_rounds=TRACK_ROUNDS,
#         )
#
#         importance = outputs['importance']
#         all_quarter_summaries.append({
#             'year': year,
#             'quarter': quarter,
#             'n_banks': len(importance),
#             'n_systemic': int(importance['is_systemically_important'].sum()),
#             'max_cascade_size': int(importance['cascade_size'].max()),
#             'max_secondary_defaults': int(importance['secondary_defaults'].max()),
#             'max_system_equity_depletion': float(importance['system_equity_depletion'].max()),
#         })
#
# full_summary_df = pd.DataFrame(all_quarter_summaries)
# display(full_summary_df)


## Notes

- This notebook uses **default-only contagion**, not random partial shocks.
- Each simulation defaults one bank with `initial_loss_frac=1.0` and `spread_without_default=False`.
- The ranking table is the main output for identifying systemically important banks.
- If `Assets` is available in the node data, the notebook also reports affected assets.